# RAG Ingestion με Vision & Caching (Google Colab GPU)

Αυτό το notebook εκτελεί την επεξεργασία (ingestion) των εκπαιδευτικών υλικών (PDF/TXT) για τα μαθήματα του RAG συστήματος χρησιμοποιώντας **GPU** για μέγιστη ταχύτητα.

### Οδηγίες Προετοιμασίας:
1. **Ενεργοποίηση GPU:** Πήγαινε στο μενού `Runtime → Change runtime type` και επίλεξε **T4 GPU**.
2. **Προσθήκη API Key:** Στα Secrets (εικονίδιο κλειδί αριστερά), πρόσθεσε το `GEMINI_API_KEY` και ενεργοποίησε το **Notebook access**.
3. **Google Drive:** Βεβαιώσου ότι τα αρχεία των μαθημάτων βρίσκονται στους φακέλους που ορίζονται στο `COURSES`.


## 1. Εγκατάσταση Απαραίτητων Βιβλιοθηκών


In [ ]:
!pip install -q sentence-transformers chromadb==0.5.9 pymupdf rank_bm25 tqdm google-genai torch pillow


## 2. Σύνδεση με το Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Το Google Drive συνδέθηκε επιτυχώς.')


## 3. Φόρτωση Gemini API Key


In [ ]:
import os
from google.colab import userdata

try:
    os.environ["GEMINI_API_KEY"] = userdata.get('GEMINI_API_KEY')
    print("Το Gemini API key φορτώθηκε με επιτυχία.")
except Exception as e:
    print("Σφάλμα κατά τη φόρτωση του API key:", e)


## 4. Ρυθμίσεις & Παράμετροι (Config)


In [ ]:
import torch

# Μοντέλα Embeddings και Vision
EMBEDDING_MODEL_NAME = 'intfloat/multilingual-e5-large-instruct'
GEMINI_VISION_MODEL  = 'gemini-2.5-flash'

# Παράμετροι Chunking & Εικόνων
MIN_CHUNK_WORDS  = 15
MIN_IMAGE_WIDTH  = 500
MIN_IMAGE_HEIGHT = 250

# Μονάδα Αποθήκευσης ChromaDB
CHROMA_PATH = "chroma_db"

# Ρυθμίσεις Μαθημάτων & Διαδρομών Google Drive
COURSES = {
    "databases": {
        "name": "Βάσεις Δεδομένων",
        "collection": "db_course",
        "bm25_index": "bm25_databases.pkl",
        "folder": "/content/drive/MyDrive/databases"
    },
    "parallel-distributed": {
        "name": "Παράλληλος και Κατανεμημένος Υπολογισμός",
        "collection": "parallel_course",
        "bm25_index": "bm25_parallel.pkl",
        "folder": "/content/drive/MyDrive/parallel-distributed"
    }
}

# Έλεγχος διαθεσιμότητας GPU
if torch.cuda.is_available():
    print(f"GPU Διαθέσιμη: {torch.cuda.get_device_name(0)}")
else:
    print("Προειδοποίηση: Δεν εντοπίστηκε GPU. Συνιστάται η αλλαγή σε GPU runtime.")


## 5. Ελληνικός Tokenizer (για BM25)


In [ ]:
import string
import unicodedata

def tokenize_greek(text):
    """Αφαιρεί τόνους, πεζά γράμματα και σημεία στίξης για το ελληνικό BM25 index."""
    text = text.lower()
    text = ''.join(c for c in unicodedata.normalize('NFD', text) if unicodedata.category(c) != 'Mn')
    for char in ['-', '«', '»', '“', '”', '·', ':', ';', '!', '?', ',', '.', '(', ')', '[', ']', '{', '}']:
        text = text.replace(char, ' ')
    text = text.translate(str.maketrans('', '', string.punctuation))
    return [t for t in text.split() if len(t) > 1]

print("Ο ελληνικός tokenizer φορτώθηκε.")


## 6. Διαχείριση Cache Περιγραφών Εικόνων


In [ ]:
import os
import json

# Αρχείο cache για αποφυγή επαναλαμβανόμενων κλήσεων στο Gemini API
CACHE_FILE = "/content/drive/MyDrive/Thesis/vision_cache.json"

if os.path.exists(CACHE_FILE):
    with open(CACHE_FILE, "r", encoding="utf-8") as f:
        vision_cache = json.load(f)
    print(f"Φορτώθηκαν {len(vision_cache)} αποθηκευμένες περιγραφές εικόνων.")
else:
    vision_cache = {}
    os.makedirs(os.path.dirname(CACHE_FILE), exist_ok=True)
    print("Δημιουργήθηκε νέο αρχείο cache.")

def save_cache():
    """Αποθηκεύει το τρέχον cache στο Google Drive."""
    with open(CACHE_FILE, "w", encoding="utf-8") as f:
        json.dump(vision_cache, f, ensure_ascii=False, indent=2)


## 7. Περιγραφή Εικόνων μέσω Gemini Vision API


In [ ]:
import hashlib
import time
import sys
import io
from google import genai
from PIL import Image

vision_client = genai.Client()

def describe_image(image_bytes: bytes, image_ext: str, page_num: int, surrounding_text: str):
    """Παράγει περιγραφή εικόνας με το Gemini API, χρησιμοποιώντας cache και διαχείριση rate limits."""
    img_hash = hashlib.md5(image_bytes).hexdigest()

    # Έλεγχος cache
    if img_hash in vision_cache:
        return vision_cache[img_hash], True

    context_hint = " ".join(surrounding_text.split()[:100]) if surrounding_text else ""
    prompt = (
        "Περίγραψε αυτή την εικόνα περιεκτικά στα Ελληνικά. "
        f"Το κείμενο της σελίδας γύρω από την εικόνα είναι: {context_hint}. "
        "Αν είναι διάγραμμα: γράψε τα components και τη ροή. Αν είναι πίνακας: κάνε transcribe όλα τα δεδομένα. Αν είναι κώδικας: αντέγραψέ τον ακριβώς. "
        "ΣΗΜΑΝΤΙΚΟ: Αν η εικόνα είναι απλά ένα εταιρικό/πανεπιστημιακό λογότυπο (logo), ένα εικονίδιο, διακοσμητικό στοιχείο ή υδατογράφημα χωρίς καμία απολύτως εκπαιδευτική αξία για το μάθημα, απάντησε ΑΥΣΤΗΡΑ ΚΑΙ ΜΟΝΟ με τη λέξη 'SKIP'."
    )

    max_retries = 3
    for attempt in range(max_retries):
        try:
            img = Image.open(io.BytesIO(image_bytes))
            response = vision_client.models.generate_content(
                model=GEMINI_VISION_MODEL,
                contents=[img, prompt]
            )
            result = response.text.strip()

            # Ενημέρωση και αποθήκευση cache
            vision_cache[img_hash] = result
            save_cache()

            return result, False

        except Exception as e:
            error_msg = str(e)
            if "429" in error_msg or "RESOURCE_EXHAUSTED" in error_msg or "503" in error_msg:
                wait_time = 65
                print(f"\n  [Rate Limit] Σελ.{page_num}: Αναμονή {wait_time}s... (Προσπάθεια {attempt+1}/{max_retries})")
                time.sleep(wait_time)
            else:
                print(f"\n  Σφάλμα Vision API (Σελ.{page_num}): {e}")
                return "", False

    print(f"\n  Αποτυχία Vision μετά από {max_retries} προσπάθειες.")
    sys.exit("ΔΙΑΚΟΠΗ: Εξαντλήθηκαν οι διαθέσιμες κλήσεις του Gemini API.")


## 8. Επεξεργασία & Chunking Αρχείων (PDF & TXT)


In [ ]:
import fitz
import glob

def process_pdf(pdf_path, current_chunk_id):
    """Εξάγει κείμενο και εικόνες από PDF και δημιουργεί chunks με metadata."""
    filename_base = os.path.basename(pdf_path)
    doc = fitz.open(pdf_path)
    total_pages = len(doc)
    page_texts = []

    for page_num in range(total_pages):
        page = doc.load_page(page_num)
        text = page.get_text("text").strip()

        # Επεξεργασία εικόνων της σελίδας
        image_list = page.get_images(full=True)
        for image_index, img in enumerate(image_list):
            try:
                xref = img[0]
                image_data = doc.extract_image(xref)

                # Φιλτράρισμα μικρών/διακοσμητικών εικόνων
                w = image_data.get("width",  0)
                h = image_data.get("height", 0)
                if w < MIN_IMAGE_WIDTH or h < MIN_IMAGE_HEIGHT or (w * h < 150000):
                    continue

                image_bytes = image_data["image"]
                image_ext   = image_data.get("ext", "png")

                vision_text, is_cached = describe_image(image_bytes, image_ext, page_num + 1, text)

                if vision_text.upper() == "SKIP" or len(vision_text.strip()) < 20:
                    continue

                if not is_cached:
                    time.sleep(10)

                text += f"\n[VISION_EXTRACT - Εικόνα {image_index+1}]: {vision_text}"

            except Exception:
                continue

        if text:
            page_texts.append((page_num, text))

    doc.close()

    # Δημιουργία chunks ανάλογα με τον ελάχιστο αριθμό λέξεων
    pdf_chunks = []
    buffer_text = ""
    buffer_start_page = 0
    for page_num, text in page_texts:
        if not buffer_text:
            buffer_text = text
            buffer_start_page = page_num
        elif len(buffer_text.split()) < MIN_CHUNK_WORDS:
            buffer_text += " " + text
        else:
            pdf_chunks.append({
                "text": buffer_text,
                "metadata": {"source": filename_base, "page": buffer_start_page, "type": "pdf"},
                "id": f"chunk_{current_chunk_id}"
            })
            current_chunk_id += 1
            buffer_text = text
            buffer_start_page = page_num

    if buffer_text and len(buffer_text.split()) >= MIN_CHUNK_WORDS:
        pdf_chunks.append({
            "text": buffer_text,
            "metadata": {"source": filename_base, "page": buffer_start_page, "type": "pdf"},
            "id": f"chunk_{current_chunk_id}"
        })
        current_chunk_id += 1

    return pdf_chunks, current_chunk_id

def process_txt(txt_path, current_chunk_id):
    """Επεξεργάζεται TXT αρχεία και παράγει chunks."""
    filename_base = os.path.basename(txt_path)
    with open(txt_path, "r", encoding="utf-8") as f:
        content = f.read()

    blocks = [b.strip() for b in content.split('\n\n') if b.strip()]
    txt_chunks = []
    for block in blocks:
        if len(block.split()) < MIN_CHUNK_WORDS:
            continue
        txt_chunks.append({
            "text": block,
            "metadata": {"source": filename_base, "type": "txt"},
            "id": f"chunk_{current_chunk_id}"
        })
        current_chunk_id += 1

    return txt_chunks, current_chunk_id


## 9. Εκτέλεση Ingestion & Δημιουργία Βάσεων (ChromaDB + BM25)


In [ ]:
import pickle
import chromadb
from tqdm.notebook import tqdm
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

def ingest_course(course_key):
    """Εκτελεί πλήρη επεξεργασία μαθήματος: chunking, GPU embeddings, ChromaDB & BM25 indexing."""
    course_info = COURSES[course_key]
    folder_path = course_info["folder"]

    if not os.path.exists(folder_path):
        print(f"Σφάλμα: Ο φάκελος {folder_path} δεν βρέθηκε!")
        return

    pdf_files = glob.glob(os.path.join(folder_path, "*.pdf"))
    txt_files = glob.glob(os.path.join(folder_path, "*.txt"))

    if not pdf_files and not txt_files:
        print(f"Δεν βρέθηκαν αρχεία PDF ή TXT στον φάκελο {folder_path}")
        return

    print(f"\n=== Έναρξη Ingestion: {course_info['name']} ===")
    print(f"Βρέθηκαν {len(pdf_files)} PDF και {len(txt_files)} TXT αρχεία.")

    all_course_chunks = []
    current_chunk_id = 0

    if pdf_files:
        for pdf_path in tqdm(pdf_files, desc="Επεξεργασία PDF"):
            chunks, current_chunk_id = process_pdf(pdf_path, current_chunk_id)
            all_course_chunks.extend(chunks)

    if txt_files:
        for txt_path in tqdm(txt_files, desc="Επεξεργασία TXT"):
            chunks, current_chunk_id = process_txt(txt_path, current_chunk_id)
            all_course_chunks.extend(chunks)

    if not all_course_chunks:
        print("Δεν δημιουργήθηκαν chunks για αυτό το μάθημα.")
        return

    texts = [c["text"] for c in all_course_chunks]
    metadata = [c["metadata"] for c in all_course_chunks]
    ids = [c["id"] for c in all_course_chunks]

    # Παραγωγή Embeddings στη GPU
    print(f"\nΦόρτωση μοντέλου Embeddings: {EMBEDDING_MODEL_NAME}...")
    embed_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

    print(f"Παραγωγή embeddings για {len(texts)} chunks στη GPU...")
    embeddings = embed_model.encode(texts, show_progress_bar=True, batch_size=64).tolist()

    # Αποθήκευση στη ChromaDB
    print("Αποθήκευση στη ChromaDB...")
    chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)
    collection_name = course_info["collection"]

    try:
        chroma_client.delete_collection(name=collection_name)
        print(f"Διαγράφηκε η παλιά συλλογή: {collection_name}")
    except Exception:
        pass

    collection = chroma_client.get_or_create_collection(name=collection_name)

    BATCH_SIZE = 5000
    for i in range(0, len(texts), BATCH_SIZE):
        end_idx = min(i + BATCH_SIZE, len(texts))
        collection.add(
            documents=texts[i:end_idx],
            embeddings=embeddings[i:end_idx],
            metadatas=metadata[i:end_idx],
            ids=ids[i:end_idx]
        )
    print(f"Αποθηκεύτηκαν {len(all_course_chunks)} chunks στη ChromaDB ({collection_name}).")

    # Κατασκευή και αποθήκευση BM25 Index
    print("Δημιουργία BM25 Index...")
    tokenized_corpus = [tokenize_greek(t) for t in texts]
    bm25 = BM25Okapi(tokenized_corpus)
    bm25_filename = course_info["bm25_index"]

    if os.path.exists(bm25_filename):
        os.remove(bm25_filename)

    with open(bm25_filename, "wb") as f:
        pickle.dump({"bm25": bm25, "chunks": texts, "metadata": metadata, "ids": ids}, f)
    print(f"Δημιουργήθηκε το αρχείο {bm25_filename}.")

    # Αποδέσμευση πόρων
    del collection
    del chroma_client
    import gc
    gc.collect()
    print("Η ChromaDB έκλεισε με ασφάλεια.")


## 10. Εκτέλεση Ingestion


In [ ]:
if not os.environ.get("GEMINI_API_KEY"):
    print("ΣΦΑΛΜΑ: Το GEMINI_API_KEY δεν έχει οριστεί στα Secrets.")
else:
    print("=== Διαθέσιμα Μαθήματα ===")
    for key, info in COURSES.items():
        print(f"  [{key}] -> {info['name']}")

    choice = input("\nΕπίλεξε μάθημα (databases / parallel-distributed / all): ").strip()

    if choice == "all":
        for key in COURSES:
            ingest_course(key)
    elif choice in COURSES:
        ingest_course(choice)
    else:
        print("Μη έγκυρη επιλογή.")


## 11. Συμπίεση & Κατέβασμα Αρχείων (ChromaDB + BM25)

Τρέξε αυτό το cell στο τέλος για να συμπιέσεις τον φάκελο `chroma_db` σε `.zip` και να τον κατεβάσεις στον υπολογιστή σου μαζί με τα `.pkl` αρχεία του BM25 index.

**Οδηγίες Τοποθέτησης στο Τοπικό Project:**
1. Αντικατάστησε τον φάκελο `chroma_db` στο τοπικό σου project με τον περιεχόμενο φάκελο του `chroma_db.zip`.
2. Τοποθέτησε τα αρχεία `.pkl` (π.χ. `bm25_databases.pkl`, `bm25_parallel.pkl`) στον κεντρικό φάκελο του τοπικού RAG project.


In [ ]:
import shutil
from google.colab import files

print("=== Συμπίεση & Κατέβασμα Αρχείων ===")

# Καθαρισμός προσωρινών αρχείων lock και συμπίεση ChromaDB
if os.path.exists(CHROMA_PATH):
    for root, dirs, files_list in os.walk(CHROMA_PATH):
        for f in files_list:
            if f.endswith(('-wal', '-shm')):
                try:
                    os.remove(os.path.join(root, f))
                except Exception:
                    pass

    shutil.make_archive("chroma_db", "zip", CHROMA_PATH)
    print("Δημιουργήθηκε το chroma_db.zip")
else:
    print("Δεν βρέθηκε φάκελος ChromaDB.")

# Κατέβασμα αρχείων zip & pkl
print("\nΈναρξη λήψης αρχείων...")
if os.path.exists("chroma_db.zip"):
    files.download("chroma_db.zip")

for key in COURSES:
    bm25_file = COURSES[key]["bm25_index"]
    if os.path.exists(bm25_file):
        files.download(bm25_file)

print("\nΗ διαδικασία ολοκληρώθηκε επιτυχώς!")
